# AI-Powered Financial Fraud Detection and Risk Analysis System

# Notebook 6: Deployment Preparation & Dashboard Integration

## Objective

This notebook prepares the trained machine learning models for deployment. It verifies model loading, creates reusable prediction functions, tests inference on sample transactions, exports deployment assets, and generates dashboard-ready files for the Streamlit application.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

print("Libraries imported successfully.")


Libraries imported successfully.


## Load Trained Models

In [2]:
iso_model = joblib.load("../models/isolation_forest.pkl")
iso_scaler = joblib.load("../models/scaler.pkl")

ae_model = load_model("../models/autoencoder.keras")
ae_scaler = joblib.load("../models/autoencoder_scaler.pkl")
ae_threshold = joblib.load("../models/threshold.pkl")

print("All deployment artifacts loaded successfully.")

All deployment artifacts loaded successfully.


## Load Processed Dataset

In [3]:
df = pd.read_csv("../data/processed/cleaned_transactions.csv")

X = df.drop(columns=["Class"])

cat_cols = X.select_dtypes(include=["object"]).columns
if len(cat_cols):
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)
X.head()

(283726, 35)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Scaled_Amount,Hour,Time_Period_Evening,Time_Period_Morning,Time_Period_Night
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0.244200,0,False,False,True
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,-0.342584,0,False,False,True
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,1.158900,0,False,False,True
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0.139886,0,False,False,True
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,-0.073813,0,False,False,True


## Reusable Prediction Functions

In [4]:
def isolation_predict(sample_df):
    scaled = iso_scaler.transform(sample_df)
    prediction = iso_model.predict(scaled)
    score = -iso_model.score_samples(scaled)

    prediction = np.where(prediction == -1, 1, 0)

    risk = (score-score.min())/(score.max()-score.min()+1e-8)*100

    return prediction, risk

def autoencoder_predict(sample_df):

    scaled = ae_scaler.transform(sample_df)

    reconstructed = ae_model.predict(scaled, verbose=0)

    error = np.mean(np.square(scaled-reconstructed), axis=1)

    prediction = (error > ae_threshold).astype(int)

    risk = (error-error.min())/(error.max()-error.min()+1e-8)*100

    return prediction, risk

## Test Inference

In [5]:
sample = X.head(10)

iso_pred, iso_risk = isolation_predict(sample)
ae_pred, ae_risk = autoencoder_predict(sample)

deployment_test = pd.DataFrame({
    "Isolation Prediction": iso_pred,
    "Isolation Risk": np.round(iso_risk,2),
    "AutoEncoder Prediction": ae_pred,
    "AutoEncoder Risk": np.round(ae_risk,2)
})

deployment_test

,Isolation Prediction,Isolation Risk,AutoEncoder Prediction,AutoEncoder Risk
0,0,24.21,0,13.46
1,0,0.88,0,0.07
2,0,95.49,0,21.58
3,0,39.96,0,100.00
4,0,26.55,0,67.72
5,0,0.00,0,0.00
6,0,17.67,0,28.75
7,0,100.00,0,53.89
8,0,34.29,0,19.05
9,0,12.70,0,0.44


## Dashboard Configuration

In [6]:
dashboard_config = {
    "project_name":"AI-Powered Financial Fraud Detection",
    "default_model":"AutoEncoder",
    "risk_levels":{
        "Low":"0-30",
        "Medium":"31-70",
        "High":"71-100"
    },
    "dashboard_pages":[
        "Home",
        "Analytics",
        "Prediction",
        "Fraud Alerts",
        "Reports",
        "Performance",
        "About"
    ]
}

dashboard_config

{'project_name': 'AI-Powered Financial Fraud Detection',
 'default_model': 'AutoEncoder',
 'risk_levels': {'Low': '0-30', 'Medium': '31-70', 'High': '71-100'},
 'dashboard_pages': ['Home',
  'Analytics',
  'Prediction',
  'Fraud Alerts',
  'Reports',
  'Performance',
  'About']}

## Deployment Verification

In [7]:
checks = {
    "Isolation Forest Loaded": iso_model is not None,
    "Isolation Scaler Loaded": iso_scaler is not None,
    "AutoEncoder Loaded": ae_model is not None,
    "AutoEncoder Scaler Loaded": ae_scaler is not None,
    "Threshold Loaded": ae_threshold is not None
}

pd.DataFrame(checks.items(), columns=["Component","Status"])

,Component,Status
0,Isolation Forest Loaded,True
1,Isolation Scaler Loaded,True
2,AutoEncoder Loaded,True
3,AutoEncoder Scaler Loaded,True
4,Threshold Loaded,True


## Export Deployment Assets

In [8]:
deployment_test.to_csv("../reports/deployment_test_results.csv", index=False)

print("Deployment verification exported successfully.")

Deployment verification exported successfully.


# Deployment Checklist

- ✔ Isolation Forest Model
- ✔ AutoEncoder Model
- ✔ Scalers
- ✔ Threshold
- ✔ Prediction Functions
- ✔ Dashboard Configuration
- ✔ Deployment Test Results


# Conclusion

The fraud detection pipeline is now fully prepared for deployment.

All trained models, preprocessing artifacts, and reusable prediction functions have been validated. The next phase is to build a professional multi-page Streamlit dashboard that loads these assets to provide:

- Live fraud prediction
- Fraud risk scoring
- Interactive analytics
- Fraud alerts
- Executive reports
- Model performance monitoring

The machine learning phase of the project is complete.
